드라이브 마운트, 폴더 생성

In [1]:
from google.colab import drive
from pathlib import Path
import os, shutil, yaml, torch

drive.mount("/content/drive")

PROJECT_PATH = Path("/content/drive/MyDrive/Single_Flower_v2")

DATASETS_DIR = PROJECT_PATH / "datasets"
RUNS_DIR = PROJECT_PATH / "training_results"
MODELS_DIR = PROJECT_PATH / "models"
VAL_DIR = PROJECT_PATH / "validation_results"
PREDICT_DIR = PROJECT_PATH / "prediction_results"
TEST_IMAGE_DIR = PROJECT_PATH / "test_images_single_flower_v2"

for p in [PROJECT_PATH, DATASETS_DIR, RUNS_DIR, MODELS_DIR, VAL_DIR, PREDICT_DIR, TEST_IMAGE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("프로젝트 경로:", PROJECT_PATH)
print("테스트 이미지 폴더:", TEST_IMAGE_DIR)

Mounted at /content/drive
프로젝트 경로: /content/drive/MyDrive/Single_Flower_v2
테스트 이미지 폴더: /content/drive/MyDrive/Single_Flower_v2/test_images_single_flower_v2


패키지 설치, GPU 확인

In [2]:
!pip install -q ultralytics roboflow pyyaml

import torch
from ultralytics import YOLO

print("CUDA 사용 가능:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU가 안 잡혔습니다. Colab 런타임을 GPU로 바꾸세요.")

!nvidia-smi

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.0/250.0 kB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 88.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 146.9 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
CUDA 사용 가능: True
GPU: Tesla T4
Sun Jun 14 12:00:05 2026       
+---------------------------------------------------------------------------

Roboflow 다운로드

In [4]:
from google.colab import userdata
from roboflow import Roboflow
from pathlib import Path
import shutil

ROBOFLOW_API_KEY = userdata.get("ROBOFLOW_API_KEY")

if not ROBOFLOW_API_KEY:
    raise ValueError("Colab Secrets에 ROBOFLOW_API_KEY를 등록하세요.")

WORKSPACE = "minseo-kim-nw4w2"
PROJECT = "single-flower"
VERSION = 1

DATASET_DRIVE_DIR = DATASETS_DIR / f"{PROJECT}-v{VERSION}-single-flower-yolo11"

FORCE_REDOWNLOAD = False

if FORCE_REDOWNLOAD and DATASET_DRIVE_DIR.exists():
    shutil.rmtree(DATASET_DRIVE_DIR)

if not (DATASET_DRIVE_DIR / "data.yaml").exists():
    rf = Roboflow(api_key=ROBOFLOW_API_KEY)
    project = rf.workspace(WORKSPACE).project(PROJECT)
    version = project.version(VERSION)

    dataset = version.download(
        "yolov11",
        location=str(DATASET_DRIVE_DIR)
    )

    print("다운로드 완료:", dataset.location)
else:
    print("이미 다운로드된 데이터셋 사용:", DATASET_DRIVE_DIR)

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to /content/drive/MyDrive/Single_Flower_v2/datasets/single-flower-v1-single-flower-yolo11 in yolov11:: 100%|██████████| 13239/13239 [02:25<00:00, 90.99it/s]

다운로드 완료: /content/drive/MyDrive/Single_Flower_v2/datasets/single-flower-v1-single-flower-yolo11


드라이브 데이터셋 로컬로 복사

In [5]:
LOCAL_DATASET_DIR = Path(f"/content/{PROJECT}_v{VERSION}_single_flower_v2")

if LOCAL_DATASET_DIR.exists():
    shutil.rmtree(LOCAL_DATASET_DIR)

shutil.copytree(DATASET_DRIVE_DIR, LOCAL_DATASET_DIR)

print("로컬 복사 완료:", LOCAL_DATASET_DIR)
print("파일 목록:", os.listdir(LOCAL_DATASET_DIR))

로컬 복사 완료: /content/single-flower_v1_single_flower_v2
파일 목록: ['README.roboflow.txt', 'test', 'valid', 'train', 'README.dataset.txt', 'data.yaml']


data.yaml single-class 검증, 수정

In [6]:
original_yaml_path = LOCAL_DATASET_DIR / "data.yaml"
fixed_yaml_path = LOCAL_DATASET_DIR / "data_fixed.yaml"

with open(original_yaml_path, "r", encoding="utf-8") as f:
    data = yaml.safe_load(f)

print("원본 data.yaml:")
print(yaml.safe_dump(data, allow_unicode=True, sort_keys=False))

fixed_data = {
    "path": str(LOCAL_DATASET_DIR),
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": 1,
    "names": ["flower"],
}

with open(fixed_yaml_path, "w", encoding="utf-8") as f:
    yaml.safe_dump(fixed_data, f, allow_unicode=True, sort_keys=False)

print("수정된 data_fixed.yaml:")
print(yaml.safe_dump(fixed_data, allow_unicode=True, sort_keys=False))

원본 data.yaml:
train: ../train/images
val: ../valid/images
test: ../test/images
nc: 1
names:
- flower
roboflow:
  workspace: minseo-kim-nw4w2
  project: single-flower
  version: 1
  license: Public Domain
  url: https://universe.roboflow.com/minseo-kim-nw4w2/single-flower/dataset/1

수정된 data_fixed.yaml:
path: /content/single-flower_v1_single_flower_v2
train: train/images
val: valid/images
test: test/images
nc: 1
names:
- flower



이미지/라벨 개수 확인

In [7]:
from pathlib import Path

def count_files(path, exts):
    path = Path(path)
    if not path.exists():
        return 0
    return len([p for p in path.iterdir() if p.suffix.lower() in exts])

img_exts = {".jpg", ".jpeg", ".png", ".webp"}
label_exts = {".txt"}

for split in ["train", "valid", "test"]:
    img_count = count_files(LOCAL_DATASET_DIR / split / "images", img_exts)
    label_count = count_files(LOCAL_DATASET_DIR / split / "labels", label_exts)

    print(f"{split}: images={img_count}, labels={label_count}")

print("YAML:", fixed_yaml_path)

train: images=6108, labels=6108
valid: images=255, labels=255
test: images=254, labels=254
YAML: /content/single-flower_v1_single_flower_v2/data_fixed.yaml


polygon -> bbox 변환

In [8]:
from pathlib import Path
import shutil

DATASET_DIR = LOCAL_DATASET_DIR

BACKUP_DIR = DATASET_DIR / "_labels_backup_before_polygon_to_bbox"

if not BACKUP_DIR.exists():
    BACKUP_DIR.mkdir(parents=True, exist_ok=True)

    for split in ["train", "valid", "test"]:
        src = DATASET_DIR / split / "labels"
        dst = BACKUP_DIR / split / "labels"

        if src.exists():
            shutil.copytree(src, dst, dirs_exist_ok=True)

    print("라벨 백업 완료:", BACKUP_DIR)
else:
    print("이미 백업 존재:", BACKUP_DIR)


def clamp01(x):
    return max(0.0, min(1.0, x))


def polygon_to_bbox(parts):
    """
    YOLO segmentation format:
    class x1 y1 x2 y2 x3 y3 ...

    YOLO detection format:
    class x_center y_center width height
    """
    cls_id = parts[0]
    coords = list(map(float, parts[1:]))

    xs = coords[0::2]
    ys = coords[1::2]

    x_min = clamp01(min(xs))
    x_max = clamp01(max(xs))
    y_min = clamp01(min(ys))
    y_max = clamp01(max(ys))

    w = x_max - x_min
    h = y_max - y_min

    if w <= 0 or h <= 0:
        return None

    x_center = x_min + w / 2
    y_center = y_min + h / 2

    return f"{cls_id} {x_center:.6f} {y_center:.6f} {w:.6f} {h:.6f}"


converted_lines = 0
kept_lines = 0
dropped_lines = 0
bad_lines = []

for split in ["train", "valid", "test"]:
    labels_dir = DATASET_DIR / split / "labels"

    if not labels_dir.exists():
        continue

    for label_path in labels_dir.glob("*.txt"):
        original_text = label_path.read_text().strip()

        if not original_text:
            continue

        new_lines = []

        for line_idx, line in enumerate(original_text.splitlines(), start=1):
            parts = line.split()

            # 이미 YOLO bbox 형식이면 그대로 유지
            if len(parts) == 5:
                cls_id = parts[0]

                if cls_id != "0":
                    parts[0] = "0"

                new_lines.append(" ".join(parts))
                kept_lines += 1
                continue

            # polygon/segmentation 형식이면 bbox로 변환
            # class + 짝수 개 좌표여야 함
            if len(parts) > 5 and (len(parts) - 1) % 2 == 0:
                parts[0] = "0"

                try:
                    bbox_line = polygon_to_bbox(parts)

                    if bbox_line is None:
                        dropped_lines += 1
                    else:
                        new_lines.append(bbox_line)
                        converted_lines += 1

                except Exception as e:
                    bad_lines.append((str(label_path), line_idx, line, str(e)))
                    dropped_lines += 1

            else:
                bad_lines.append((str(label_path), line_idx, line, "알 수 없는 라벨 형식"))
                dropped_lines += 1

        label_path.write_text("\n".join(new_lines) + ("\n" if new_lines else ""))

print("기존 bbox 유지:", kept_lines)
print("polygon -> bbox 변환:", converted_lines)
print("drop된 라벨:", dropped_lines)
print("여전히 이상한 라벨:", len(bad_lines))

for item in bad_lines[:20]:
    print(item)

라벨 백업 완료: /content/single-flower_v1_single_flower_v2/_labels_backup_before_polygon_to_bbox
기존 bbox 유지: 54425
polygon -> bbox 변환: 70
drop된 라벨: 0
여전히 이상한 라벨: 0


모든 class id가 0인지 검사

In [9]:
bad_labels = []
empty_labels = 0
total_boxes = 0

for split in ["train", "valid", "test"]:
    labels_dir = LOCAL_DATASET_DIR / split / "labels"

    if not labels_dir.exists():
        continue

    for label_path in labels_dir.glob("*.txt"):
        text = label_path.read_text().strip()

        if not text:
            empty_labels += 1
            continue

        for line_idx, line in enumerate(text.splitlines(), start=1):
            parts = line.split()

            if len(parts) != 5:
                bad_labels.append((str(label_path), line_idx, line, "컬럼 수 이상"))
                continue

            cls_id = parts[0]
            total_boxes += 1

            if cls_id != "0":
                bad_labels.append((str(label_path), line_idx, line, "class id가 0이 아님"))

print("총 bbox 수:", total_boxes)
print("빈 label 파일 수:", empty_labels)
print("문제 label 수:", len(bad_labels))

for item in bad_labels[:20]:
    print(item)

if bad_labels:
    raise ValueError("class id 또는 label 형식 문제가 있습니다. 위 출력 확인하세요.")

총 bbox 수: 54495
빈 label 파일 수: 10
문제 label 수: 0


학습 시작

In [ ]:
from ultralytics import YOLO
from pathlib import Path
import torch

assert torch.cuda.is_available(), "GPU가 안 잡혔습니다. Colab 런타임을 GPU로 바꾸세요."

DATA_YAML = str(fixed_yaml_path)

MODEL_NAME = "yolo11m.pt"
IMG_SIZE = 1024
BATCH_SIZE = 4

RUN_NAME = f"single_flower_yolo11m_v{VERSION}_{IMG_SIZE}_fitblack_3x"

model = YOLO(MODEL_NAME)

results = model.train(
    data=DATA_YAML,

    epochs=120,
    patience=30,

    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    workers=4,
    device=0,

    project=str(RUNS_DIR),
    name=RUN_NAME,
    exist_ok=False,

    pretrained=True,
    optimizer="auto",
    cos_lr=True,

    hsv_h=0.0,
    hsv_s=0.0,
    hsv_v=0.0,

    degrees=0.0,
    translate=0.0,
    scale=0.0,
    shear=0.0,
    perspective=0.0,

    fliplr=0.0,
    flipud=0.0,

    mosaic=0.0,
    mixup=0.0,
    copy_paste=0.0,

    cache=False,
    amp=True,
    plots=True,
    save_period=10,

    seed=42,
    deterministic=True,
    single_cls=True,
)

Ultralytics 8.4.67 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/single-flower_v1_single_flower_v2/data_fixed.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=120, erasing=0.4, exist_ok=False, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.0, hsv_s=0.0, hsv_v=0.0, imgsz=1024, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11m.pt, momentum=0.937, mosaic=0.0, multi_scale=0.0, name=single_flower_yolo11m_v1_1024_fitblack_3x, nbs=64, nms=False, opset=None, opti

학습 결과 위치 확인

In [ ]:
RUN_DIR = RUNS_DIR / RUN_NAME
BEST_PT = RUN_DIR / "weights" / "best.pt"
LAST_PT = RUN_DIR / "weights" / "last.pt"

print("RUN_DIR:", RUN_DIR)
print("best.pt exists:", BEST_PT.exists(), BEST_PT)
print("last.pt exists:", LAST_PT.exists(), LAST_PT)

중간에 끊겼을 때 이어서

In [ ]:
from ultralytics import YOLO
from pathlib import Path

LAST_PT = Path("/content/drive/MyDrive/Single_Flower_v2/training_results/여기에-run-folder/weights/last.pt")

print("last.pt exists:", LAST_PT.exists())
print(LAST_PT)

model = YOLO(str(LAST_PT))
results = model.train(resume=True)

validation / test 평가

In [ ]:
from ultralytics import YOLO

BEST_PT = RUN_DIR / "weights" / "best.pt"

model = YOLO(str(BEST_PT))

val_results = model.val(
    data=str(fixed_yaml_path),
    split="val",
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    device=0,
    project=str(VAL_DIR),
    name=f"{RUN_NAME}_val",
    exist_ok=True,
)

test_results = model.val(
    data=str(fixed_yaml_path),
    split="test",
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    device=0,
    project=str(VAL_DIR),
    name=f"{RUN_NAME}_test",
    exist_ok=True,
)

최종 모델 복사

In [ ]:
import shutil
from pathlib import Path

FINAL_MODEL = MODELS_DIR / f"{RUN_NAME}_best.pt"

shutil.copy2(BEST_PT, FINAL_MODEL)

print("최종 모델 복사 완료:")
print(FINAL_MODEL)

테스트

In [ ]:
from ultralytics import YOLO
from pathlib import Path
from IPython.display import Image, display

BEST_PT = RUN_DIR / "weights" / "best.pt"

TEST_IMAGE_DIR = Path("/content/drive/MyDrive/Single_Flower_v2/test_images_single_flower")
PREDICT_DIR = Path("/content/drive/MyDrive/Single_Flower_v2/prediction_results")

model = YOLO(str(BEST_PT))

PREDICT_NAME = f"{RUN_NAME}_conf005_iou075"

results = model.predict(
    source=str(TEST_IMAGE_DIR),
    imgsz=IMG_SIZE,
    conf=0.05,
    iou=0.75,
    max_det=500,
    device=0,

    save=True,
    save_txt=True,
    save_conf=True,

    project=str(PREDICT_DIR),
    name=PREDICT_NAME,
    exist_ok=True,
)

RESULT_DIR = PREDICT_DIR / PREDICT_NAME
print("결과 저장:", RESULT_DIR)

예측 이미지 확인

In [ ]:
image_paths = []

for ext in ["*.jpg", "*.jpeg", "*.png", "*.webp"]:
    image_paths.extend(RESULT_DIR.glob(ext))

print("결과 이미지 개수:", len(image_paths))

for img_path in image_paths[:30]:
    print(img_path.name)
    display(Image(filename=str(img_path), width=700))

In [ ]:
from ultralytics import YOLO
from pathlib import Path

BEST_PT = RUN_DIR / "weights" / "best.pt"
TEST_IMAGE_DIR = Path("/content/drive/MyDrive/Single_Flower_v2/test_images_single_flower")
PREDICT_DIR = Path("/content/drive/MyDrive/Single_Flower_v2/prediction_results")

model = YOLO(str(BEST_PT))

settings = [
    {"conf": 0.03, "iou": 0.75, "name": "conf003_iou075"},
    {"conf": 0.05, "iou": 0.75, "name": "conf005_iou075"},
    {"conf": 0.08, "iou": 0.75, "name": "conf008_iou075"},
    {"conf": 0.10, "iou": 0.75, "name": "conf010_iou075"},
    {"conf": 0.05, "iou": 0.85, "name": "conf005_iou085"},
]

for s in settings:
    results = model.predict(
        source=str(TEST_IMAGE_DIR),
        imgsz=IMG_SIZE,
        conf=s["conf"],
        iou=s["iou"],
        max_det=500,
        device=0,

        save=True,
        save_txt=True,
        save_conf=True,

        project=str(PREDICT_DIR),
        name=f"{RUN_NAME}_{s['name']}",
        exist_ok=True,
    )

    print("saved:", s)

In [ ]:
from pathlib import Path
from PIL import Image as PILImage, ImageDraw, ImageFont
from IPython.display import display

PREDICT_DIR = Path("/content/drive/MyDrive/Single_Flower_v2/prediction_results")

result_dirs = {
    "conf003_iou075": PREDICT_DIR / f"{RUN_NAME}_conf003_iou075",
    "conf005_iou075": PREDICT_DIR / f"{RUN_NAME}_conf005_iou075",
    "conf008_iou075": PREDICT_DIR / f"{RUN_NAME}_conf008_iou075",
    "conf010_iou075": PREDICT_DIR / f"{RUN_NAME}_conf010_iou075",
    "conf005_iou085": PREDICT_DIR / f"{RUN_NAME}_conf005_iou085",
}

def list_image_files(folder):
    folder = Path(folder)
    paths = []
    for ext in ["*.jpg", "*.jpeg", "*.png", "*.webp"]:
        paths.extend(folder.glob(ext))
    return sorted(paths)

base_key = "conf005_iou075"
base_images = list_image_files(result_dirs[base_key])
image_names = [p.name for p in base_images]

print("비교 가능한 이미지 개수:", len(image_names))
print(image_names[:10])

In [ ]:
def make_comparison_grid(image_name, width_each=360):
    panels = []

    for label, folder in result_dirs.items():
        img_path = folder / image_name

        if not img_path.exists():
            continue

        img = PILImage.open(img_path).convert("RGB")

        ratio = width_each / img.width
        new_h = int(img.height * ratio)
        img = img.resize((width_each, new_h))

        header_h = 38
        canvas = PILImage.new("RGB", (width_each, new_h + header_h), "white")
        canvas.paste(img, (0, header_h))

        draw = ImageDraw.Draw(canvas)
        draw.text((8, 10), label, fill=(0, 0, 0))

        panels.append(canvas)

    if not panels:
        return None

    total_w = sum(p.width for p in panels)
    max_h = max(p.height for p in panels)

    grid = PILImage.new("RGB", (total_w, max_h), "white")

    x = 0
    for p in panels:
        grid.paste(p, (x, 0))
        x += p.width

    return grid


# 앞에서부터 10장 비교
for image_name in image_names[:10]:
    print(image_name)
    grid = make_comparison_grid(image_name, width_each=320)
    if grid:
        display(grid)